## Data loading and preparation

Import dataset from kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("frabbisw/facial-age")

print("Path to dataset files:", path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
models_dir = '/content/drive/MyDrive/models'

Check the structure of the dataset

In [ ]:
from pathlib import Path

path = Path(path)

def hierarchy(root: Path):
    return { child: hierarchy(child) for child in root.iterdir() } if root.is_dir() else None

path_dict = hierarchy(path)
path_dict

In [ ]:
path_dict = dict(list(path_dict.items())[0:])

path_dict

In [ ]:
data = {}

for directory in path_dict:
    for age in path_dict[directory]:
        for image in path_dict[directory][age]:
            if image.is_file():
                data[image] = age.name

print(data)

In [ ]:
import pandas as pd


df = pd.DataFrame(data = {'file' : data.keys(), 'age' : data.values()})
print(df.sample(5))

After manually inspecting the dataset, it was decided to drop some of the examples as they were either corrupted files, different body parts than face or having the wrong age.

In [ ]:
print(len(df))
files_to_drop = [3829, 4313, 7034, 7326, 9378, 1490,]
for filename in files_to_drop:
    filename_with_extension = f"{filename}.png"
    for index, row in df.iterrows():
        if filename_with_extension in str(row.file):
            print(row.file)
            df = df.drop(index)

print(len(df))

In [ ]:
df.to_csv('facial-age.csv')

We can categorize ages into bins to simplify classification task.
To start with it we can simple create bins with approximately equal number of examples in each.
The downside of it is that we won't have as precise age in case of wider bins (where number of examples were low for some ages). And the upside is that we will have approximarely equal number of examples in each bin, which in theory will allow us to predict a category with more accuracy.

In [ ]:
df_sorted = df.sort_values('age', ascending=False)
print(df_sorted.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

ages_plot = sns.countplot(data=df_sorted['age'])
ages_plot.set(yticklabels=[110] + [' '] * 97 + [1]) # sorry for hardcoded number :)
ages_plot.set_title('Distribution of ages')
plt.show()

In [ ]:
df['age'] = df['age'].astype(int)
df['age_bins'] = pd.qcut(x=df['age'], q=8, precision=0)
print(df.sample(5))

In [ ]:
sns.countplot(data=df['age_bins']).set_title('Distribution of groups of ages')


In [ ]:
df['age_bins'] = df['age_bins'].astype(str)
df.info()

In [ ]:
import numpy as np
train, validate, test = \
              np.split(df.sample(frac=1),
                       [int(.75*len(df)), int(.9*len(df))])

print(len(train), len(validate), len(test))

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Add a 'split' column to each DataFrame to identify their origin
train_plot = train.copy()
train_plot['split'] = 'train'

validate_plot = validate.copy()
validate_plot['split'] = 'validate'

test_plot = test.copy()
test_plot['split'] = 'test'

# Concatenate the DataFrames
combined_df = pd.concat([train_plot, validate_plot, test_plot])

# Calculate counts for stacked bar chart
counts_df = combined_df.groupby(['age_bins', 'split']).size().unstack(fill_value=0)

# Plot the stacked bar chart
plt.figure(figsize=(12, 7))
counts_df.plot(kind='bar', stacked=True, ax=plt.gca())
plt.title('Stacked Distribution of Age Groups Across Splits')
plt.xlabel('Age Bins')
plt.ylabel('Number of Images')
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
plt.legend(title='Dataset Split')
plt.tight_layout()
plt.show()

The proportions might be not exactly equal, but we can attribute it to some randomness in sampling.

Giving up and starting using LLMs to have a chance to get things done before the deadline.

In [ ]:
!pip install tensorflow

# Part 3. Backbone models and fine tuning

## Split training set

In [ ]:
training_block_1 = train.sample(frac=0.5, random_state=42)
training_block_2 = train.drop(training_block_1.index)

print(f"Training block 1 size: {len(training_block_1)}")
print(f"Training block 2 size: {len(training_block_2)}")

There was a note about using LLMs, but I've lost it in unequal fight with Colab.

In [ ]:
training_block_1.to_csv(models_dir + 'training_block_1.cvs')
training_block_2.to_csv(models_dir + 'training_block_2.cvs')

## Autoencoder Modelling: For Block 1 Images

### Task
Define a wrapper function `autoencoder_generator` that takes an existing `ImageDataGenerator` iterator and yields `(image, image)` pairs instead of `(image, label)` pairs, as autoencoders require the input to be the target. Then, using the existing `train`, `validate`, and `test` dataframes and the `datagen` (with `rescale=1./255`), create new base generators for RGB images (64x64) and wrap them to produce `train_gen_auto`, `val_gen_auto`, and `test_gen_auto`.

In [ ]:
# 1. Define generator wrapper for Autoencoder (Target = Input)
def autoencoder_generator(generator):
    while True:
        # Extract batch. flow_from_dataframe yields (x, y)
        # We disregard y (labels) and yield (x, x)
        x, y = next(generator)
        yield x, x

# 2. Create Base Generators (using existing 'datagen')
print("Setting up Base Generators for Autoencoder...")
train_gen_base = datagen.flow_from_dataframe(
    dataframe=training_block_1,
    x_col='file',
    y_col='age_bins',
    target_size=(64, 64),
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb'
)

val_gen_base = datagen.flow_from_dataframe(
    dataframe=validate,
    x_col='file',
    y_col='age_bins',
    target_size=(64, 64),
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb'
)

test_gen_base = datagen.flow_from_dataframe(
    dataframe=test,
    x_col='file',
    y_col='age_bins',
    target_size=(64, 64),
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb'
)

# 3. Create Autoencoder Iterators
train_gen_auto = autoencoder_generator(train_gen_base)
val_gen_auto = autoencoder_generator(val_gen_base)
test_gen_auto = autoencoder_generator(test_gen_base)

print("Autoencoder generators initialized.")

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.models import Model
import os
import pickle

# 1. Define Autoencoder Architecture
input_img = Input(shape=(64, 64, 3))

# Encoder
x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
encoded = MaxPooling2D((2, 2), padding='same')(x)

# Decoder
x = Conv2D(128, (3, 3), activation='relu', padding='same')(encoded)
x = UpSampling2D((2, 2))(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='mean_squared_error')
autoencoder.summary()

# 2. Train the Model
# Calculate steps per epoch manually since we are using generators derived from dataframes
steps_per_epoch = len(train) // 32
validation_steps = len(validate) // 32

model_filename = os.path.join(models_dir, 'autoencoder.keras')
history_filename = os.path.join(models_dir, 'autoencoder_history.pkl')

if os.path.exists(model_filename) and os.path.exists(history_filename):
    print("Loading existing autoencoder model and history...")
    autoencoder = keras.models.load_model(model_filename)
    with open(history_filename, 'rb') as f:
        history_auto = pickle.load(f)
else:
    print("Training autoencoder...")
    history_obj = autoencoder.fit(
        train_gen_auto,
        epochs=15,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_gen_auto,
        validation_steps=validation_steps,
        verbose=1
    )
    history_auto = history_obj.history

    # Save model and history
    autoencoder.save(model_filename)
    with open(history_filename, 'wb') as f:
        pickle.dump(history_auto, f)
    print("Saved autoencoder model and history.")

### Task
Visualize the autoencoder's performance by selecting a batch of test images from `test_gen_auto`, generating their reconstructions, and plotting them side-by-side. Then, add a discussion on why MSE was chosen as the loss function and what other metrics like MAE or SSIM could indicate. Finally, provide a summary of the autoencoder construction and an assessment of the image quality.

In [ ]:
import matplotlib.pyplot as plt

# 1. Retrieve a single batch of test images
# The generator yields (x, x), so we take the first element
x_test, _ = next(test_gen_auto)

# 2. Generate reconstructed images
decoded_imgs = autoencoder.predict(x_test)

# 3. Visualize a subset of images
n = 10  # Number of images to display
plt.figure(figsize=(20, 4))
for i in range(n):
    # Display original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i])
    plt.title("Original")
    plt.axis("off")

    # Display reconstruction
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i])
    plt.title("Reconstructed")
    plt.axis("off")

plt.suptitle("Autoencoder Reconstruction Results")
plt.show()

We can see that the reconstructed image quality is somewhat worse than the original. I think that is a normal thing and attribute it to pooling and upsampling, where we could lose some information.

### Discussion and Summary

**Choice of Loss Function (MSE):**
Mean Squared Error (MSE) is a standard loss function for autoencoders performing image reconstruction. It calculates the average squared difference between the pixel values of the original and reconstructed images. By penalizing larger errors more severely, MSE encourages the model to capture the general structure and color distribution effectively. However, it operates on a pixel-by-pixel basis, which can sometimes lead to blurry reconstructions as the model 'averages' out high-frequency details to minimize the overall error.

**Alternative Metrics:**
*   **MAE (Mean Absolute Error):** Calculates the average absolute difference. It is less sensitive to outliers than MSE and can sometimes result in slightly sharper edges, though it still doesn't explicitly capture structural information.
*   **SSIM (Structural Similarity Index):** A perceptual metric that measures the similarity between two images based on luminance, contrast, and structure. Unlike pixel-wise metrics (MSE/MAE), SSIM correlates better with human perception of image quality. A high SSIM indicates that the reconstructed image preserves the structural information of the original well.

**Autoencoder Construction Summary:**
The model is a Convolutional Autoencoder designed for 64x64 RGB images.
*   **Encoder:** Compresses the input into a lower-dimensional latent representation using a series of `Conv2D` layers (relu activation) followed by `MaxPooling2D` for downsampling.
*   **Decoder:** Reconstructs the image from the latent representation using `Conv2D` layers and `UpSampling2D` to recover the spatial dimensions, ending with a `sigmoid` activation to output pixel values in the [0, 1] range.

**Assessment of Image Quality:**
Visually, the reconstructed images likely capture the global facial features, pose, and skin tone of the original images. However, fine details (like individual hair strands or skin texture) might be smoothed out or slightly blurry. This is typical for simple autoencoders trained with MSE, as the bottleneck forces the model to prioritize the most significant features required to approximate the input.

I'm curious to try SSIM loss function if I'll have time.

## Transfer learning (Block 2)

### Task
Use `training_block_2` to create a new `train_gen_transfer` data generator with the same settings as before (RGB, 64x64, sparse). Then, build a transfer learning model by extracting the encoder layers from the trained `autoencoder` (up to the bottleneck), followed by a `Flatten` layer, a `Dense` layer with 64 units (ReLU), and a final `Dense` output layer (Softmax). Freeze the encoder layers so their weights do not change during training. Train this new model for 10 epochs using `train_gen_transfer` and the existing `val_gen_base`. Finally, evaluate the model on `test_gen_base` and generate a bar chart comparing its accuracy and loss against the original RGB baseline model (available in `results_channels['rgb']`).

In [ ]:
print("Setting up Transfer Learning Generator (Block 2)...")
train_gen_transfer = datagen.flow_from_dataframe(
    dataframe=training_block_2,
    x_col='file',
    y_col='age_bins',
    target_size=(64, 64),
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb'
)

print("Transfer learning generator initialized.")

In [ ]:
# 1. Extract Encoder
# The bottleneck is at index 6 (MaxPooling2D after 3rd Conv block)
encoder_output = autoencoder.layers[6].output
encoder_model = keras.Model(inputs=autoencoder.input, outputs=encoder_output, name='encoder')

# 2. Freeze Encoder
encoder_model.trainable = False

# 3. Build Transfer Learning Model
model_transfer = models.Sequential([
    encoder_model,
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(8, activation='softmax')
], name='transfer_learning_model')

# 4. Compile
model_transfer.compile(optimizer='adam',
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])

# 5. Summary
model_transfer.summary()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
from tensorflow import keras

# 1. Train or Load Model
model_filename = os.path.join(models_dir, 'model_transfer.keras')
history_filename = os.path.join(models_dir, 'history_transfer.pkl')

if os.path.exists(model_filename) and os.path.exists(history_filename):
    print("Loading existing transfer learning model and history...")
    model_transfer = keras.models.load_model(model_filename)
    with open(history_filename, 'rb') as f:
        history_transfer = pickle.load(f)
else:
    print("Training transfer learning model...")
    history_obj = model_transfer.fit(
        train_gen_transfer,
        epochs=10,
        validation_data=val_gen_base,
        verbose=1
    )
    history_transfer = history_obj.history

    # Save
    model_transfer.save(model_filename)
    with open(history_filename, 'wb') as f:
        pickle.dump(history_transfer, f)
    print("Saved transfer learning model and history.")

# 2. Evaluate on Test Data
print("Evaluating transfer learning model on test data...")
test_loss_transfer, test_acc_transfer = model_transfer.evaluate(test_gen_base, verbose=0)
print(f"Transfer Model - Test Accuracy: {test_acc_transfer:.4f}")
print(f"Transfer Model - Test Loss: {test_loss_transfer:.4f}")

# 3. Comparison Visualization
print(f"\n{'='*60}\nComparison: Transfer Learning vs Baseline RGB\n{'='*60}")

# Retrieve Baseline Results
# Assuming results_channels['rgb'] exists from previous steps
if 'results_channels' in locals() and 'rgb' in results_channels:
    baseline_acc = results_channels['rgb']['acc']
    baseline_loss = results_channels['rgb']['loss']
else:
    # Fallback if variable lost, using hardcoded approximate values or re-evaluating would be needed,
    # but assuming context is preserved.
    print("Warning: Baseline results not found in memory. Using placeholders if necessary.")
    baseline_acc = 0.0
    baseline_loss = 0.0

labels = ['Baseline RGB', 'Transfer Learning']
accuracies = [baseline_acc, test_acc_transfer]
losses = [baseline_loss, test_loss_transfer]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(10, 6))
rects1 = plt.bar(x - width/2, accuracies, width, label='Test Accuracy')
rects2 = plt.bar(x + width/2, losses, width, label='Test Loss')

plt.ylabel('Scores')
plt.title('Test Metrics: Baseline vs Transfer Learning')
plt.xticks(x, labels)
plt.legend()
plt.bar_label(rects1, padding=3, fmt='%.4f')
plt.bar_label(rects2, padding=3, fmt='%.4f')
plt.show()

# 4. Plot Learning Curves for Transfer Model
plt.figure(figsize=(14, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_transfer['accuracy'], label='Train Accuracy')
plt.plot(history_transfer['val_accuracy'], linestyle='--', label='Val Accuracy')
plt.title('Transfer Learning Model - Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_transfer['loss'], label='Train Loss')
plt.plot(history_transfer['val_loss'], linestyle='--', label='Val Loss')
plt.title('Transfer Learning Model - Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

Haha, it didn't work out the way I thought it will. I mean, it's around the same model? We've taken only encoder, so we didn't attempt to reconstruct the image, so we didn't lose some information on it. I wonder why results for transfer learning a worse then? Only because there were less examples to train on (block 2 only)?

Asking Gemini.


It says that additionaly we've compiled autoencoder with 'MSE' loss function as opposed to 'sparse_categorical_crossentropy' for the original model. That seems like it might bu true, right.

## Backbone Model Re-Use

Prompt: What pre-trained generic image models are there?

Responce:
There are several generic, pre-trained image processing models available in libraries like Keras, typically trained on the massive ImageNet dataset. Common examples include:

VGG16 / VGG19: Known for their simple architecture (stack of convolutional layers), though relatively heavy in terms of parameters.
ResNet50 (and variants): Uses residual connections (skip connections) to allow for much deeper networks without vanishing gradient issues.
InceptionV3: Uses parallel convolutional blocks of different sizes (kernels) to capture features at various scales.
MobileNet / MobileNetV2: Optimized for efficiency and speed, ideal for resource-constrained environments.
EfficientNet: Scales depth, width, and resolution uniformly for state-of-the-art accuracy-to-efficiency trade-offs.

Prompt: What is the difference between VGG16 and VGG19?

Repsonce: The primary difference is the depth of the network. VGG16 has 16 weight layers (13 convolutional layers and 3 fully connected layers), while VGG19 has 19 weight layers (16 convolutional layers and 3 fully connected layers).

VGG19 essentially adds three more convolutional layers to the architecture. While this makes VGG19 capable of learning slightly more complex features, it also makes it computationally heavier and sometimes harder to train on smaller datasets compared to VGG16. For many tasks, VGG16 provides a better balance between performance and efficiency.


Now I think 16 layers will be enough for a quick experiment, providing we do not do any critical research here.

### Task
Create a new training data generator `train_gen_vgg` using the `training_block_1` dataframe with the same settings as the RGB baseline (rescale=1./255, target_size=(64, 64), batch_size=32, class_mode='sparse', color_mode='rgb').

Then, build a VGG16-based model:
1. Load the VGG16 model pre-trained on ImageNet with `include_top=False` and `input_shape=(64, 64, 3)`.
2. Freeze the base VGG16 layers so they are not trainable.
3. Add a classification head consisting of a `Flatten` layer, a `Dense` layer with 64 units (ReLU activation), and a final `Dense` layer with 8 units (Softmax activation).

Train this model for 10 epochs using `train_gen_vgg` and the existing `val_gen_base`. Evaluate the trained model on `test_gen_base`.

Finally, generate a bar chart comparing the Test Accuracy and Test Loss of this VGG16 model against the previously obtained results for the RGB Baseline (`results_channels['rgb']`) and the Autoencoder Transfer model (`test_acc_transfer`, `test_loss_transfer`).

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Ensure datagen is available
if 'datagen' not in locals() and 'datagen' not in globals():
    datagen = ImageDataGenerator(rescale=1./255)

print("Setting up VGG16 Training Generator (Block 1)...")
train_gen_vgg = datagen.flow_from_dataframe(
    dataframe=training_block_1,
    x_col='file',
    y_col='age_bins',
    target_size=(64, 64),
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb'
)

print("VGG16 training generator initialized.")

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
import os
import pickle
from tensorflow import keras

# 1. Instantiate VGG16 Base
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(64, 64, 3))

# 2. Freeze Base Model
base_model.trainable = False

# 3. Build Sequential Model
model_vgg16 = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(8, activation='softmax')
], name='vgg16_transfer')

# 4. Compile
model_vgg16.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

model_vgg16.summary()

# 5. Train or Load
model_filename = os.path.join(models_dir, 'model_vgg16.keras')
history_filename = os.path.join(models_dir, 'history_vgg16.pkl')

if os.path.exists(model_filename) and os.path.exists(history_filename):
    print("Loading existing VGG16 model and history...")
    model_vgg16 = keras.models.load_model(model_filename)
    with open(history_filename, 'rb') as f:
        history_vgg16 = pickle.load(f)
else:
    print("Training VGG16 model...")
    history_obj = model_vgg16.fit(
        train_gen_vgg,
        epochs=10,
        validation_data=val_gen_base,
        verbose=1
    )
    history_vgg16 = history_obj.history

    # Save
    model_vgg16.save(model_filename)
    with open(history_filename, 'wb') as f:
        pickle.dump(history_vgg16, f)
    print("Saved VGG16 model and history.")

### Task
Evaluate the `model_vgg16` on `test_gen_base` to obtain the final Test Accuracy and Test Loss. Retrieve the stored metrics for the Baseline RGB model (`results_channels['rgb']`) and the Autoencoder Transfer model (`test_acc_transfer`, `test_loss_transfer`). Create a grouped bar chart comparing these metrics side-by-side. Finally, provide a text summary comparing the VGG16 Transfer model to the RGB Baseline, focusing on model complexity, training behavior, and performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Evaluate VGG16 on Test Data
print("Evaluating VGG16 model on test data...")
test_loss_vgg16, test_acc_vgg16 = model_vgg16.evaluate(test_gen_base, verbose=0)

# 2. Gather Metrics for Comparison
# Baseline RGB
if 'results_channels' in locals() and 'rgb' in results_channels:
    baseline_acc = results_channels['rgb']['acc']
    baseline_loss = results_channels['rgb']['loss']
else:
    baseline_acc = 0.0
    baseline_loss = 0.0
    print("Warning: Baseline RGB results not found.")

# Autoencoder Transfer (variables should exist from previous cells)
if 'test_acc_transfer' not in locals():
    test_acc_transfer = 0.0
    test_loss_transfer = 0.0
    print("Warning: Transfer Learning results not found.")

# 3. Plotting Comparison
labels = ['Baseline RGB', 'Autoencoder Transfer', 'VGG16 Transfer']
accuracies = [baseline_acc, test_acc_transfer, test_acc_vgg16]
losses = [baseline_loss, test_loss_transfer, test_loss_vgg16]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(12, 6))
rects1 = plt.bar(x - width/2, accuracies, width, label='Test Accuracy')
rects2 = plt.bar(x + width/2, losses, width, label='Test Loss')

plt.ylabel('Scores')
plt.title('Model Comparison: Baseline vs Transfer Learning (Autoencoder & VGG16)')
plt.xticks(x, labels)
plt.legend()

plt.bar_label(rects1, padding=3, fmt='%.4f')
plt.bar_label(rects2, padding=3, fmt='%.4f')

plt.tight_layout()
plt.show()

# 4. Print Summary
print(f"Baseline RGB Model:       Test Accuracy = {baseline_acc:.4f}, Test Loss = {baseline_loss:.4f}")
print(f"Autoencoder Transfer:     Test Accuracy = {test_acc_transfer:.4f}, Test Loss = {test_loss_transfer:.4f}")
print(f"VGG16 Transfer Model:     Test Accuracy = {test_acc_vgg16:.4f}, Test Loss = {test_loss_vgg16:.4f}")

Well, this is not ideal :)

The upside is that now I better understand the downside of LLMs, using this analogy.
The size makes the model more suitable for a broad set of tasks, but it doesn't mean it will perform better on more specific tasks.